In [ ]:
import pymongo
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, date
import os
import pprint

load_dotenv(override=True)

### setup connection

In [ ]:
mongo_uri = os.getenv("mongo_uri")
# mongo_uri_local = os.getenv("mongo_uri_local")

try:
    client = pymongo.MongoClient(mongo_uri)
except NameError:
    print("mongo_uri not available, trying mongo_uri_local")
    client=pymongo.MongoClient(mongo_uri_local)

from pymongo.uri_parser import parse_uri
# print(parse_uri(mongo_uri_local)["options"])
# print(repr(mongo_uri_local))



client.list_database_names()
db = client["klapp-prod"]

# Data ideas

## 1. Students with duplicated parents (parents share the same name)

For each student, look up their linked parents and flag the student if two *different* parent accounts (different `_id`) resolve to the same name.

In [ ]:
list(db["user"].find({"role": "parent"}).limit(1))

In [ ]:
db["student"].find_one()

In [ ]:
pipeline = [
    # no role filter needed - the "student" collection only contains students
    {"$lookup": {
        "from": "user",
        "localField": "parents",
        "foreignField": "_id",
        "as": "parent_docs",
    }},
    {"$addFields": {
        "parent_first_name": "$parent_docs.first_name",
        "parent_last_name": "$parent_docs.last_name",
    }},
    {"$match": {
        # flag when there are more parent docs than distinct (first_name, last_name) pairs
        "$expr": {
            "$gt": [
                {"$size": "$parent_docs"},
                {"$size": {"$setUnion": [
                    {"$map": {
                        "input": "$parent_docs",
                        "as": "p",
                        "in": {"$concat": ["$$p.first_name", " ", "$$p.last_name"]},
                    }},
                    [],
                ]}},
            ]
        }
    }},
    {"$project": {
        "first_name": 1,
        "last_name": 1,
        "parent_docs._id": 1,
        "parent_first_name": 1,
        "parent_last_name": 1,
    }},
]

df = pd.DataFrame(list(db["student"].aggregate(pipeline)))

In [ ]:
df

In [ ]:
pipeline = [
    {"$lookup": {
        "from": "user",
        "localField": "parents",
        "foreignField": "_id",
        "as": "parent_docs",
    }},
    {"$match": {
        "$expr": {
            "$gt": [
                {"$size": "$parent_docs"},
                {"$size": {"$setUnion": [
                    {"$map": {
                        "input": "$parent_docs",
                        "as": "p",
                        "in": {"$concat": ["$$p.first_name", " ", "$$p.last_name"]},
                    }},
                    [],
                ]}},
            ]
        }
    }},
    {"$group": {
        # siblings from the same family share the same "parents" array, so sorting+grouping
        # on it collapses them into a single incident instead of one per child
        "_id": {"$sortArray": {"input": "$parents", "sortBy": 1}},  # requires MongoDB 5.2+
        "children": {"$push": {"_id": "$_id", "first_name": "$first_name", "last_name": "$last_name"}},
        "num_children": {"$sum": 1},
    }},
]

results = list(db["student"].aggregate(pipeline))
print(f"distinct duplicate-parent incidents: {len(results)}")
print(f"students affected (sum of num_children): {sum(r['num_children'] for r in results)}")
results[:10]

### 1c. Check whether flagged "duplicate parents" actually have role == parent

The lookup above joins whatever `_id`s appear in a student's `parents` array without checking `role`. If one of those `_id`s belongs to a `teacher`-role account (same person, different role-account) rather than a second `parent`-role account, it's not really a duplicate-account bug — it's one person with two legitimate role-accounts. This checks the `role` field of every parent_doc involved in a flagged duplicate.

In [ ]:
pipeline = [
    {"$lookup": {
        "from": "user",
        "localField": "parents",
        "foreignField": "_id",
        "as": "parent_docs",
    }},
    {"$match": {
        "$expr": {
            "$gt": [
                {"$size": "$parent_docs"},
                {"$size": {"$setUnion": [
                    {"$map": {
                        "input": "$parent_docs",
                        "as": "p",
                        "in": {"$concat": ["$$p.first_name", " ", "$$p.last_name"]},
                    }},
                    [],
                ]}},
            ]
        }
    }},
    {"$project": {
        "first_name": 1,
        "last_name": 1,
        "parent_docs._id": 1,
        "parent_docs.first_name": 1,
        "parent_docs.last_name": 1,
        "parent_docs.role": 1,
    }},
]

df_roles = pd.DataFrame(list(db["student"].aggregate(pipeline)))
df_roles["parent_roles"] = df_roles["parent_docs"].apply(lambda docs: [d.get("role") for d in docs])
df_roles["has_non_parent_role"] = df_roles["parent_roles"].apply(
    lambda roles: any(r != ["parent"] for r in roles)
)

print(f"flagged rows total: {len(df_roles)}")
print(f"flagged rows where a parent_doc role != ['parent']: {df_roles['has_non_parent_role'].sum()}")
df_roles[df_roles["has_non_parent_role"]].head(10)

## 2. Parents with the same name as a student

Finds parent docs whose name matches the name of some student doc (independent of whether they're linked as family) — useful to spot mixed-up roles or accidental duplicates.

In [ ]:
pipeline = [
    {"$match": {"user_type": "parent"}},  # TODO: confirm field/value that identifies a parent doc
    {"$lookup": {
        "from": "users",  # TODO: confirm collection name
        "let": {"pname": "$full_name"},  # TODO: parent's name field
        "pipeline": [
            {"$match": {"user_type": "student"}},  # TODO
            {"$match": {"$expr": {"$eq": ["$full_name", "$$pname"]}}},  # TODO: student's name field
        ],
        "as": "matching_students",
    }},
    {"$match": {"matching_students": {"$ne": []}}},
    {"$project": {
        "full_name": 1,
        "matching_students._id": 1,
        "matching_students.full_name": 1,
    }},
]

list(db.users.aggregate(pipeline))

## 3. Users who are both student and parent

Two possible schemas — use whichever matches reality:

- **3a**: a single user doc has a `roles` array containing both values.
- **3b**: role is a separate doc per role, linked by a shared identity field (e.g. email).

In [ ]:
# 3a: single doc, roles stored as an array field
query = {
    "roles": {"$all": ["student", "parent"]},  # TODO: confirm field is "roles" (array), not "user_type" (scalar)
}

list(db.users.find(query))

In [ ]:
# 3b: one doc per role, grouped by a shared identity field (e.g. email / national id)
pipeline = [
    {"$group": {
        "_id": "$email",  # TODO: field that uniquely identifies the *person* across role-docs
        "roles": {"$addToSet": "$user_type"},  # TODO: role field name
        "doc_ids": {"$push": "$_id"},
    }},
    {"$match": {"roles": {"$all": ["student", "parent"]}}},
]

list(db.users.aggregate(pipeline))